In [25]:
from epo.tipdata.patstat import PatstatClient
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
import time

patstat = PatstatClient(env='PROD')

def q(sql):
    res = patstat.sql_query(sql, use_legacy_sql=False)
    return pd.DataFrame(res)

TU = """
WITH tu AS (
    SELECT DISTINCT a.appln_id, a.docdb_family_id, a.appln_filing_year,
           a.appln_auth, a.granted, a.nb_applicants
    FROM tls201_appln a
    JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
    JOIN tls206_person p ON pa.person_id = p.person_id
    WHERE pa.applt_seq_nr > 0 AND a.docdb_family_id > 0
      AND a.appln_filing_year BETWEEN 2013 AND 2024
      AND (
        LOWER(p.person_name) LIKE '%technische universit%dortmund%'
        OR LOWER(p.person_name) LIKE '%technische universitaet dortmund%'
        OR LOWER(p.person_name) LIKE '%univ dortmund%'
        OR LOWER(p.person_name) LIKE '%tech universit%dortmund%'
        OR LOWER(p.person_name) LIKE '%uni dortmund%'
        OR LOWER(p.person_name) LIKE '%technical univ%dortmund%'
        OR LOWER(p.person_name) LIKE '%universit%dortmund%'
      )
      AND LOWER(p.person_name) NOT LIKE '%elmos%'
      AND LOWER(p.person_name) NOT LIKE '%goch%'
)
"""

display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=Source+Serif+4:wght@400;600;700&family=JetBrains+Mono:wght@400;500;600&display=swap');
:root {
  --teal: #14b8a6; --teal-dark: #0d9488; --teal-glow: rgba(20,184,166,0.12);
  --blue: #3b82f6; --blue-light: rgba(59,130,246,0.12);
  --bg: #f4f5f7; --card: #fff; --text: #1e293b; --text-sec: #64748b;
  --text-dim: #94a3b8; --border: #e2e8f0;
  --grad-1: #0f172a; --grad-2: #1e293b; --grad-3: #0d1f3c;
}
.rpt-cover {
  background: linear-gradient(135deg, var(--grad-1), var(--grad-2), var(--grad-3));
  color: #fff; padding: 56px 40px 96px; text-align: center; border-radius: 12px;
  position: relative; overflow: hidden; margin-bottom: 0;
}
.rpt-cover::after {
  content:''; position:absolute; top:-30%; right:-10%; width:600px; height:600px;
  background: radial-gradient(circle, var(--teal-glow), transparent 70%);
  border-radius:50%; pointer-events:none;
}
.rpt-cover h1 { font-family:'Source Serif 4',Georgia,serif; font-size:2.4em; font-weight:700; line-height:1.15; margin:0 0 12px; }
.rpt-cover .sub { font-size:1.1em; color:#cbd5e1; font-weight:300; margin-bottom:24px; }
.rpt-cover .meta { font-family:'JetBrains Mono',monospace; font-size:0.75em; color:#94a3b8; }
.kpi-strip {
  display:grid; grid-template-columns:repeat(5,1fr); gap:14px;
  margin:-48px auto 32px; position:relative; z-index:2; max-width:960px;
}
.kpi-card {
  background:var(--card); border:1px solid var(--border); border-radius:12px;
  padding:22px 16px; text-align:center; box-shadow:0 2px 8px rgba(0,0,0,0.06);
}
.kpi-card .num { font-family:'JetBrains Mono',monospace; font-size:1.9em; font-weight:600; color:var(--text); }
.kpi-card .lbl { font-size:0.72em; color:var(--text-sec); margin-top:4px; text-transform:uppercase; letter-spacing:0.04em; }
.kpi-card .chg {
  font-family:'JetBrains Mono',monospace; font-size:0.68em; padding:2px 8px;
  border-radius:10px; display:inline-block; margin-top:6px;
}
.chg.up { background:#f0fdf4; color:#16a34a; }
.chg.down { background:#fef2f2; color:#dc2626; }
.chg.flat { background:#f1f5f9; color:var(--text-sec); }
.rpt-card {
  background:var(--card); border:1px solid var(--border); border-radius:12px;
  padding:32px 36px; box-shadow:0 1px 3px rgba(0,0,0,0.04); margin:24px auto; max-width:960px;
}
.rpt-card h2 { font-family:'Source Serif 4',Georgia,serif; font-size:1.35em; font-weight:600; margin:0 0 4px; color:var(--text); }
.rpt-card .desc { font-size:0.88em; color:var(--text-sec); margin-bottom:20px; }
.insight {
  border-left:3px solid var(--teal-dark); background:#f0fdfa;
  padding:14px 18px; border-radius:0 8px 8px 0; margin:18px 0; font-size:0.88em; line-height:1.7;
}
.insight strong { color:var(--teal-dark); }
.rpt-tbl { width:100%; border-collapse:collapse; font-size:0.85em; font-family:'Inter',sans-serif; }
.rpt-tbl th {
  text-align:left; font-weight:600; color:var(--text-sec); font-size:0.8em;
  text-transform:uppercase; letter-spacing:0.04em; padding:10px 12px; border-bottom:2px solid var(--border);
}
.rpt-tbl td { padding:9px 12px; border-bottom:1px solid #f1f5f9; color:var(--text); }
.rpt-tbl tr:hover td { background:#fafbfc; }
.rpt-tbl .n { text-align:right; font-family:'JetBrains Mono',monospace; font-weight:500; }
.period-tag { font-family:'JetBrains Mono',monospace; font-size:0.7em; padding:2px 10px; border-radius:10px; font-weight:500; }
.period-tag.p1 { background:var(--blue-light); color:var(--blue); }
.period-tag.p2 { background:var(--teal-glow); color:var(--teal-dark); }
.compare-grid { display:grid; grid-template-columns:1fr 1fr; gap:18px; margin:18px 0; }
.compare-card { border:1px solid var(--border); border-radius:10px; padding:20px 22px; }
.compare-card.p1 { border-top:3px solid var(--blue); }
.compare-card.p2 { border-top:3px solid var(--teal); }
.compare-card h3 { font-family:'JetBrains Mono',monospace; font-size:0.82em; margin:0 0 14px; }
.cm { display:flex; justify-content:space-between; padding:5px 0; border-bottom:1px solid #f8fafc; font-size:0.88em; }
.cm .v { font-family:'JetBrains Mono',monospace; font-weight:600; }
footer.rpt {
  text-align:center; font-family:'JetBrains Mono',monospace; font-size:0.72em;
  color:var(--text-dim); line-height:1.8; margin:32px auto; max-width:960px;
}
footer.rpt a { color:var(--teal-dark); text-decoration:none; }
</style>
"""))

In [26]:
HTML(
    """
<div class="rpt-cover">
  <h1>Patent Portfolio in Transition</h1>
  <div class="sub">TU Dortmund — Comparative Analysis 2013–2024<br>
  Pre- and Post-Excellence Start-Up Center</div>
  <div class="meta">PATSTAT Global · Autumn 2025 · Report: 2026-02-23 · mtc.berlin</div>
</div>
"""
)

In [27]:
df2 = q(TU + """
SELECT
    CASE WHEN appln_filing_year BETWEEN 2013 AND 2018 THEN '2013-2018'
         ELSE '2019-2024' END AS period,
    COUNT(DISTINCT docdb_family_id) AS families,
    COUNT(DISTINCT appln_id) AS applications,
    COUNT(DISTINCT CASE WHEN nb_applicants > 1 THEN docdb_family_id END) AS co_filed,
    COUNT(DISTINCT CASE WHEN granted = 'Y' THEN appln_id END) AS granted
FROM tu GROUP BY 1 ORDER BY 1
""")
r1, r2 = df2.iloc[0], df2.iloc[1]
tf = int(r1.families + r2.families)
ta = int(r1.applications + r2.applications)

def chg(a, b):
    if a == 0: return '<span class="chg up">NEW</span>'
    pct = 100*(b-a)/a
    cls = 'up' if pct > 2 else ('down' if pct < -2 else 'flat')
    sign = '+' if pct > 0 else ''
    return f'<span class="chg {cls}">{sign}{pct:.0f}%</span>'

kpis = [
    (tf, 'Families', chg(r1.families, r2.families)),
    (ta, 'Applications', chg(r1.applications, r2.applications)),
    (f'{100*int(r1.co_filed+r2.co_filed)/tf:.0f}%', 'Co-Filing Rate', chg(r1.co_filed, r2.co_filed)),
    (int(r1.co_filed + r2.co_filed), 'Co-Applications', ''),
    (int(r1.granted + r2.granted), 'Granted Apps', chg(r1.granted, r2.granted)),
]
cards = ''.join(f'<div class="kpi-card"><div class="num">{v}</div><div class="lbl">{l}</div>{c}</div>' for v,l,c in kpis)
HTML(f'<div style="display:grid;grid-template-columns:repeat(5,1fr);gap:14px;max-width:960px;margin:20px auto 32px">{cards}</div>')

In [28]:
df3 = q(
    TU
    + """
SELECT appln_filing_year AS year,
    COUNT(DISTINCT docdb_family_id) AS families,
    COUNT(DISTINCT appln_id) AS applications,
    COUNT(DISTINCT CASE WHEN nb_applicants > 1 THEN docdb_family_id END) AS co_filed,
    COUNT(DISTINCT CASE WHEN granted = 'Y' THEN appln_id END) AS granted
FROM tu GROUP BY 1 ORDER BY 1
"""
)

In [29]:
blue, teal, gray = "#3b82f6", "#14b8a6", "#64748b"
colors = [blue if y <= 2018 else teal for y in df3["year"]]

fig = go.Figure()
fig.add_bar(
    x=df3["year"],
    y=df3["families"],
    name="Families",
    marker_color=colors,
    text=df3["families"],
    textposition="outside",
)
fig.add_scatter(
    x=df3["year"],
    y=df3["applications"],
    name="Applications",
    mode="lines+markers",
    line=dict(color=gray, width=2),
    marker=dict(size=5),
)
fig.add_vline(
    x=2018.5,
    line_dash="dash",
    line_color="#ef4444",
    annotation_text="ESC Launch",
    annotation_font_size=11,
)
fig.update_layout(
    title=None,
    height=380,
    bargap=0.3,
    font=dict(family="Inter, sans-serif", size=12),
    xaxis=dict(dtick=1, title=None),
    yaxis=dict(title=None),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(t=40, b=30, l=40, r=20),
    plot_bgcolor="white",
    paper_bgcolor="white",
)

p1m = df3[df3["year"] <= 2018]["families"].mean()
p2m = df3[df3["year"] > 2018]["families"].mean()
pk = df3.loc[df3["families"].idxmax()]

display(
    HTML(
        f"""
<div class="rpt-card">
  <h2>Filing Activity Over Time</h2>
  <div class="desc">DOCDB families and applications per filing year, 2013–2024.</div>
</div>
"""
    )
)
fig.show()
display(
    HTML(
        f"""
<div style="max-width:960px;margin:0 auto">
<div class="insight">
  <strong>Key Finding:</strong> Filing volume remained stable — average
  <strong>{p1m:.1f}</strong> families/year (2013–18) vs <strong>{p2m:.1f}</strong> (2019–24).
  Peak: <strong>{int(pk.year)}</strong> with {int(pk.families)} families.
</div>
</div>
"""
    )
)

In [30]:
def metric_row(label, v1, v2, fmt="d", suffix=""):
    v1d = f"{v1:{fmt}}{suffix}"
    v2d = f"{v2:{fmt}}{suffix}"
    return (
        f'<div class="cm"><span>{label}</span><span class="v">{v1d}</span></div>',
        f'<div class="cm"><span>{label}</span><span class="v">{v2d}</span></div>',
    )


co1_pct = 100 * int(r1.co_filed) / int(r1.families)
co2_pct = 100 * int(r2.co_filed) / int(r2.families)
gr1_pct = 100 * int(r1.granted) / int(r1.applications)
gr2_pct = 100 * int(r2.granted) / int(r2.applications)

metrics = [
    ("Patent Families", int(r1.families), int(r2.families), "d", ""),
    ("Applications", int(r1.applications), int(r2.applications), "d", ""),
    ("Co-Filed Families", int(r1.co_filed), int(r2.co_filed), "d", ""),
    ("Co-Filing Rate", co1_pct, co2_pct, ".1f", "%"),
    ("Granted Apps", int(r1.granted), int(r2.granted), "d", ""),
    ("Grant Rate", gr1_pct, gr2_pct, ".1f", "%"),
]
left = right = ""
for label, v1, v2, fmt, suf in metrics:
    l, r = metric_row(label, v1, v2, fmt, suf)
    left += l
    right += r

HTML(
    f"""
<div class="rpt-card">
  <h2>Period Comparison</h2>
  <div class="desc">Side-by-side KPI dashboard — pre- vs post-Excellence Start-Up Center.</div>
  <div class="compare-grid">
    <div class="compare-card p1">
      <h3><span class="period-tag p1">2013 – 2018</span></h3>
      {left}
    </div>
    <div class="compare-card p2">
      <h3><span class="period-tag p2">2019 – 2024</span></h3>
      {right}
    </div>
  </div>
</div>
"""
)

In [31]:
df1 = q(
    """
SELECT p.person_name, p.han_name, p.person_ctry_code AS ctry,
    COUNT(DISTINCT a.docdb_family_id) AS families,
    MIN(a.appln_filing_year) AS earliest, MAX(a.appln_filing_year) AS latest
FROM tls206_person p
JOIN tls207_pers_appln pa ON p.person_id = pa.person_id
JOIN tls201_appln a ON pa.appln_id = a.appln_id
WHERE pa.applt_seq_nr > 0 AND a.docdb_family_id > 0
  AND a.appln_filing_year BETWEEN 2013 AND 2024
  AND LOWER(p.person_name) LIKE '%dortmund%'
  AND (LOWER(p.person_name) LIKE '%universit%' OR LOWER(p.person_name) LIKE '%univ %'
    OR LOWER(p.person_name) LIKE '%tu %' OR LOWER(p.person_name) LIKE '%tech%'
    OR LOWER(p.person_name) LIKE '% uni %')
GROUP BY p.person_name, p.han_name, p.person_ctry_code
ORDER BY families DESC
"""
)

rows = ""
for _, r in df1.iterrows():
    rows += f"""<tr>
      <td>{r.person_name}</td><td>{r.han_name}</td><td>{r.ctry}</td>
      <td class="n">{int(r.families)}</td><td class="n">{int(r.earliest)}</td><td class="n">{int(r.latest)}</td>
    </tr>"""

HTML(
    f"""
<div class="rpt-card">
  <h2>Applicant Name Variants</h2>
  <div class="desc">All <code>person_name</code> spellings found for TU Dortmund in PATSTAT.
    Note: most map to <code>han_name</code> "TECHNISCHE UNIVERSITAT MUNCHEN" (incorrect).</div>
  <table class="rpt-tbl">
    <thead><tr><th>Person Name</th><th>HAN Name</th><th>Ctry</th><th class="n">Families</th><th class="n">Earliest</th><th class="n">Latest</th></tr></thead>
    <tbody>{rows}</tbody>
  </table>
  <div class="insight"><strong>Methodology:</strong> We use <code>person_name</code> matching instead of
    <code>han_name</code> because PATSTAT's harmonization incorrectly maps most TU Dortmund
    variants to "TECHNISCHE UNIVERSITAT MUNCHEN". {len(df1)} distinct spellings found.</div>
</div>
"""
)

Person Name,HAN Name,Ctry,Families,Earliest,Latest
TECHNISCHE UNIVERSITÄT DORTMUND,TECHNISCHE UNIVERSITAT MUNCHEN,DE,94,2013,2024
Technische Universität Dortmund,TECHNISCHE UNIVERSITAT MUNCHEN,DE,56,2013,2023
"Technische Universität Dortmund, Körperschaft des öffentlichen Rechts",TECHNISCHE UNIV DORTMUND KORPERSCHAFT DES OFFENTLICHEN RECHTS,DE,25,2020,2024
TECHNISCHE UNIVERSITAET DORTMUND,TECHNISCHE UNIVERSITAT MUNCHEN,DE,9,2013,2022
TECHNISCHE UNIVERSITAT DORTMUND,TECHNISCHE UNIVERSITAT MUNCHEN,DE,3,2016,2019
TECHNISCHE UNIVERSITU T DORTMUND,TECHNISCHE UNIVERSITU T DORTMUND,,2,2022,2023
Technische Universitat Dortmund,TECHNISCHE UNIVERSITAT MUNCHEN,DE,1,2019,2019
TECHNISCHE UNIVERSIT?T DORTMUND,TECHNISCHE UNIV T DORTMUND,DE,1,2022,2022
TECHNISCHE UNIVERSITÄT DORTMUND KÖR,TECHNISCHE UNIV DORTMUND KOR,DE,1,2024,2024
"Technische Universität Dortmund, Körperschaft des Öffentlichen Rechts",TECHNISCHE UNIV DORTMUND KORPERSCHAFT DES OFFENTLICHEN RECHTS,DE,1,2024,2024


In [32]:
# LAST cell — hides all code, prompts, and text output after Run All
HTML("""
<footer class="rpt">
  Data: EPO PATSTAT Global, Autumn 2025 (BigQuery) · Coverage: ~95% through mid-2024; 2024 preliminary<br>
  Analysis: Arne Krüger · <a href="https://depa.tech">depa.tech</a> · Moving Targets Consulting GmbH, Berlin<br>
  Generated from Jupyter Notebook on the EPO Technology Intelligence Platform
</footer>
<style>
/* Hide code inputs */
div.input, div.jp-InputArea, div.jp-Cell-inputWrapper { display: none !important; }
/* Hide cell numbers / prompts */
div.prompt, .jp-InputPrompt, .jp-OutputPrompt { display: none !important; }
/* Hide text/print output (timing lines, etc.) */
div.output_text, div.output_stderr,
div.output_stdout, pre.output_text,
.jp-OutputArea-output[data-mime-type="text/plain"],
.jp-OutputArea-output[data-mime-type="application/vnd.jupyter.stderr"] { display: none !important; }
/* Remove cell margins for clean flow */
div.cell, .jp-Cell { margin-left: 0 !important; }
</style>
""")